In [1]:
import pandas as pd
import sqlite3
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')

# Paths
project_path = '/content/drive/MyDrive/olist_project'
db_path = f'{project_path}/data/processed/olist.db'

# Connect to the SQLite database from Stage 2
conn = sqlite3.connect(db_path)

# Verify all 9 tables are there
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print("Tables in database:")
print(tables)

Mounted at /content/drive
Tables in database:
                   name
0             customers
1           geolocation
2           order_items
3        order_payments
4         order_reviews
5                orders
6              products
7               sellers
8  category_translation


In [2]:
# Load products and category translations
products = pd.read_sql("SELECT * FROM products;", conn)
translations = pd.read_sql("SELECT * FROM category_translation;", conn)

print(f"Products: {len(products)} rows")
print(f"Translations: {len(translations)} rows")
print(f"Unique product categories in products table: {products['product_category_name'].nunique()}")
print(f"Categories with translations available: {translations['product_category_name'].nunique()}")

Products: 32951 rows
Translations: 71 rows
Unique product categories in products table: 73
Categories with translations available: 71


In [3]:
# Left join translations onto products
products_translated = products.merge(
    translations,
    on='product_category_name',
    how='left'
)

# Find categories that didn't get a translation
missing = products_translated[products_translated['product_category_name_english'].isna()]
print(f"Rows with missing English translation: {len(missing)}")
print(f"Categories without translation: {missing['product_category_name'].unique()}")

Rows with missing English translation: 623
Categories without translation: [None 'pc_gamer' 'portateis_cozinha_e_preparadores_de_alimentos']


In [4]:
manual_translations = {
    'portateis_cozinha_e_preparadores_de_alimentos': 'kitchen_portable_appliances',
    'pc_gamer': 'pc_gamer',  # already English
}

products_translated['product_category_name_english'] = products_translated.apply(
    lambda row: manual_translations.get(row['product_category_name'], row['product_category_name_english'])
    if pd.isna(row['product_category_name_english']) else row['product_category_name_english'],
    axis=1
)

# Verify no more missing
still_missing = products_translated[products_translated['product_category_name_english'].isna()]
print(f"Remaining without English: {len(still_missing)}")

Remaining without English: 610


In [9]:
# How many products have no category at all?
no_category = products_translated[products_translated['product_category_name'].isna()]
print(f"Products with no category in source: {len(no_category)}")

# How many have a Portuguese category but no English translation?
has_pt_no_en = products_translated[
    products_translated['product_category_name'].notna() &
    products_translated['product_category_name_english'].isna()
]
print(f"Products with Portuguese category but no English translation: {len(has_pt_no_en)}")

# What are the untranslated Portuguese categories?
print(f"\nUntranslated Portuguese categories:")
print(has_pt_no_en['product_category_name'].value_counts())

Products with no category in source: 610
Products with Portuguese category but no English translation: 0

Untranslated Portuguese categories:
Series([], Name: count, dtype: int64)


In [6]:
products_translated.to_sql('products_clean', conn, if_exists='replace', index=False)
print("✓ Saved products_clean to database")

✓ Saved products_clean to database


In [7]:
# Look at status distribution
status = pd.read_sql("""
    SELECT order_status, COUNT(*) AS n
    FROM orders
    GROUP BY order_status
    ORDER BY n DESC
""", conn)
print(status)

  order_status      n
0    delivered  96478
1      shipped   1107
2     canceled    625
3  unavailable    609
4     invoiced    314
5   processing    301
6      created      5
7     approved      2


In [8]:
# Create a delivered-only orders table
delivered_orders_sql = """
CREATE TABLE IF NOT EXISTS orders_delivered AS
SELECT *
FROM orders
WHERE order_status = 'delivered'
  AND order_delivered_customer_date IS NOT NULL
  AND order_purchase_timestamp IS NOT NULL;
"""

cursor = conn.cursor()
cursor.execute("DROP TABLE IF EXISTS orders_delivered;")
cursor.execute(delivered_orders_sql)
conn.commit()

# Verify
count = pd.read_sql("SELECT COUNT(*) AS n FROM orders_delivered;", conn)
print(f"Delivered orders with valid dates: {count.iloc[0,0]}")

Delivered orders with valid dates: 96470


In [10]:
# Add derived columns to orders_delivered using SQL date arithmetic
# SQLite stores these as text, so we use julianday() to convert to numeric days

cursor.execute("DROP TABLE IF EXISTS orders_with_delivery;")

derived_orders_sql = """
CREATE TABLE orders_with_delivery AS
SELECT
    *,
    -- Total delivery time in days
    ROUND(julianday(order_delivered_customer_date) - julianday(order_purchase_timestamp), 1) AS delivery_days,

    -- Delay vs estimated (positive = late, negative = early)
    ROUND(julianday(order_delivered_customer_date) - julianday(order_estimated_delivery_date), 1) AS delay_vs_estimate_days,

    -- Was the order late?
    CASE
        WHEN julianday(order_delivered_customer_date) > julianday(order_estimated_delivery_date) THEN 1
        ELSE 0
    END AS was_late
FROM orders_delivered;
"""

cursor.execute(derived_orders_sql)
conn.commit()

sample = pd.read_sql("""
    SELECT order_id, delivery_days, delay_vs_estimate_days, was_late
    FROM orders_with_delivery
    LIMIT 5;
""", conn)
print(sample)

# What % were late?
late_pct = pd.read_sql("""
    SELECT
        ROUND(100.0 * SUM(was_late) / COUNT(*), 1) AS pct_late,
        ROUND(AVG(delivery_days), 1) AS avg_delivery_days
    FROM orders_with_delivery;
""", conn)
print(late_pct)

                           order_id  delivery_days  delay_vs_estimate_days  \
0  e481f51cbdc54678b7cc49136f2d6af7            8.4                    -7.1   
1  53cdb2fc8bc7dce0b6741e2150273451           13.8                    -5.4   
2  47770eb9100c2d0c44946d9cf07ec65d            9.4                   -17.2   
3  949d5b44dbf5de918fe9c16f97b45f8a           13.2                   -13.0   
4  ad21c59c0840e6cb83a9ceb5573f8159            2.9                    -9.2   

   was_late  
0         0  
1         0  
2         0  
3         0  
4         0  
   pct_late  avg_delivery_days
0       8.1               12.6


In [11]:
# Build order_revenue table: one row per order with total value
cursor.execute("DROP TABLE IF EXISTS order_revenue;")

revenue_sql = """
CREATE TABLE order_revenue AS
SELECT
    order_id,
    COUNT(*) AS items_count,
    ROUND(SUM(price), 2) AS revenue,
    ROUND(SUM(freight_value), 2) AS freight,
    ROUND(SUM(price + freight_value), 2) AS total_value
FROM order_items
GROUP BY order_id;
"""

cursor.execute(revenue_sql)
conn.commit()

# Verify
preview = pd.read_sql("""
    SELECT * FROM order_revenue
    ORDER BY revenue DESC
    LIMIT 5;
""", conn)
print(preview)

                           order_id  items_count  revenue  freight  \
0  03caa2c082116e1d31e67e9ae3700499            8  13440.0   224.08   
1  736e1922ae60d0d6a89247b851902527            4   7160.0   114.88   
2  0812eb902a67711a1cb742b3cdaa65ae            1   6735.0   194.31   
3  fefacc66af859508bf1a7934eab1e97f            1   6729.0   193.21   
4  f5136e38d1a14a4dbd87dff67da82701            1   6499.0   227.66   

   total_value  
0     13664.08  
1      7274.88  
2      6929.31  
3      6922.21  
4      6726.66  


In [12]:
cursor.execute("DROP TABLE IF EXISTS master_orders;")

master_sql = """
CREATE TABLE master_orders AS
SELECT
    o.order_id,
    c.customer_unique_id,
    c.customer_state,
    c.customer_city,
    o.order_purchase_timestamp,
    o.order_delivered_customer_date,
    o.delivery_days,
    o.delay_vs_estimate_days,
    o.was_late,
    r.revenue,
    r.items_count,
    r.total_value,
    rv.review_score
FROM orders_with_delivery o
JOIN customers c ON o.customer_id = c.customer_id
LEFT JOIN order_revenue r ON o.order_id = r.order_id
LEFT JOIN order_reviews rv ON o.order_id = rv.order_id;
"""

cursor.execute(master_sql)
conn.commit()

# Verify
count = pd.read_sql("SELECT COUNT(*) AS n FROM master_orders;", conn)
print(f"Master orders rows: {count.iloc[0,0]}")

preview = pd.read_sql("SELECT * FROM master_orders LIMIT 5;", conn)
print(preview)

# Check for nulls
nulls = pd.read_sql("""
SELECT
    SUM(CASE WHEN revenue IS NULL THEN 1 ELSE 0 END) AS null_revenue,
    SUM(CASE WHEN review_score IS NULL THEN 1 ELSE 0 END) AS null_review_score,
    SUM(CASE WHEN customer_unique_id IS NULL THEN 1 ELSE 0 END) AS null_customer
FROM master_orders;
""", conn)
print(nulls)

Master orders rows: 96999
                           order_id                customer_unique_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  7c396fd4830fd04220f754e42b4e5bff   
1  53cdb2fc8bc7dce0b6741e2150273451  af07308b275d755c9edb36a90c618231   
2  47770eb9100c2d0c44946d9cf07ec65d  3a653a41f6f9fc3d2a113cf8398680e8   
3  949d5b44dbf5de918fe9c16f97b45f8a  7c142cf63193a1473d2e66489a9ae977   
4  ad21c59c0840e6cb83a9ceb5573f8159  72632f0f9dd73dfee390c9b22eb56dd6   

  customer_state            customer_city order_purchase_timestamp  \
0             SP                sao paulo      2017-10-02 10:56:33   
1             BA                barreiras      2018-07-24 20:41:37   
2             GO               vianopolis      2018-08-08 08:38:49   
3             RN  sao goncalo do amarante      2017-11-18 19:28:06   
4             SP              santo andre      2018-02-13 21:18:39   

  order_delivered_customer_date  delivery_days  delay_vs_estimate_days  \
0           2017-10-10 21:25:13         

In [13]:
# Check for duplicates in master_orders
dups = pd.read_sql("""
SELECT order_id, COUNT(*) AS n
FROM master_orders
GROUP BY order_id
HAVING COUNT(*) > 1
ORDER BY n DESC
LIMIT 10;
""", conn)
print(f"Orders with duplicate rows: {len(dups)}")
print(dups.head() if len(dups) > 0 else "No duplicates")

Orders with duplicate rows: 10
                           order_id  n
0  df56136b8031ecd28e200bb18e6ddb2e  3
1  c88b1d1b157a9999ce368f218a407141  3
2  8e17072ec97ce29f0e1f111e598b0c85  3
3  03c939fd7fd3b38f8485a0f95798f1f6  3
4  ffaabba06c9d293a3c614e0515ddbabc  2


In [14]:
# Check if these duplicate orders have multiple reviews
check = pd.read_sql("""
    SELECT order_id, COUNT(*) AS review_count
    FROM order_reviews
    WHERE order_id IN (
        'df56136b8031ecd28e200bb18e6ddb2e',
        'c88b1d1b157a9999ce368f218a407141',
        '8e17072ec97ce29f0e1f111e598b0c85',
        '03c939fd7fd3b38f8485a0f95798f1f6',
        'ffaabba06c9d293a3c614e0515ddbabc'
    )
    GROUP BY order_id;
""", conn)
print(check)

                           order_id  review_count
0  03c939fd7fd3b38f8485a0f95798f1f6             3
1  8e17072ec97ce29f0e1f111e598b0c85             3
2  c88b1d1b157a9999ce368f218a407141             3
3  df56136b8031ecd28e200bb18e6ddb2e             3
4  ffaabba06c9d293a3c614e0515ddbabc             2


In [15]:
#  Verify
dups = pd.read_sql("""
    SELECT order_id, COUNT(*) AS n
    FROM master_orders
    GROUP BY order_id
    HAVING COUNT(*) > 1;
""", conn)
print(f"Orders with duplicate rows after fix: {len(dups)}")

# Confirming orders now appear exactly once
check = pd.read_sql("""
    SELECT order_id, COUNT(*) AS n
    FROM master_orders
    WHERE order_id IN (
        'df56136b8031ecd28e200bb18e6ddb2e',
        'c88b1d1b157a9999ce368f218a407141',
        '8e17072ec97ce29f0e1f111e598b0c85',
        '03c939fd7fd3b38f8485a0f95798f1f6',
        'ffaabba06c9d293a3c614e0515ddbabc'
    )
    GROUP BY order_id;
""", conn)
print(check)

Orders with duplicate rows after fix: 525
                           order_id  n
0  03c939fd7fd3b38f8485a0f95798f1f6  3
1  8e17072ec97ce29f0e1f111e598b0c85  3
2  c88b1d1b157a9999ce368f218a407141  3
3  df56136b8031ecd28e200bb18e6ddb2e  3
4  ffaabba06c9d293a3c614e0515ddbabc  2


In [16]:
# Investigate where the duplicates are coming from
# Check 1: Are the 525 duplicates the same 5 orders we knew about, or new ones?
all_dups = pd.read_sql("""
    SELECT order_id, COUNT(*) AS n
    FROM master_orders
    GROUP BY order_id
    HAVING COUNT(*) > 1
    ORDER BY n DESC;
""", conn)
print(f"Total orders with duplicates: {len(all_dups)}")
print(f"\nDistribution of duplicate counts:")
print(all_dups['n'].value_counts())
print(f"\nFirst 10 duplicate orders:")
print(all_dups.head(10))

Total orders with duplicates: 525

Distribution of duplicate counts:
n
2    521
3      4
Name: count, dtype: int64

First 10 duplicate orders:
                           order_id  n
0  df56136b8031ecd28e200bb18e6ddb2e  3
1  c88b1d1b157a9999ce368f218a407141  3
2  8e17072ec97ce29f0e1f111e598b0c85  3
3  03c939fd7fd3b38f8485a0f95798f1f6  3
4  ffaabba06c9d293a3c614e0515ddbabc  2
5  ff850ba359507b996e8b2fbb26df8d03  2
6  ff763b73e473d03c321bcd5a053316e8  2
7  fe041ba1c9f54016432fa6ee91709dbc  2
8  fd95ae805c63c534f1a64589e102225e  2
9  fd61441ba2a7b57e6342862e779b10b0  2


In [20]:
# === Full rebuild of master_orders with review deduplication ===

# Step 1: Drop the existing (broken) table
cursor.execute("DROP TABLE IF EXISTS master_orders;")
conn.commit()
print("✓ Dropped old master_orders")

# Step 2: Rebuild with CTE that picks the latest review per order
master_sql = """
CREATE TABLE master_orders AS
WITH latest_reviews AS (
    SELECT
        order_id,
        review_score,
        review_creation_date,
        ROW_NUMBER() OVER (
            PARTITION BY order_id
            ORDER BY review_creation_date DESC
        ) AS rn
    FROM order_reviews
)
SELECT
    o.order_id,
    c.customer_unique_id,
    c.customer_state,
    c.customer_city,
    o.order_purchase_timestamp,
    o.order_delivered_customer_date,
    o.delivery_days,
    o.delay_vs_estimate_days,
    o.was_late,
    r.revenue,
    r.items_count,
    r.total_value,
    rv.review_score
FROM orders_with_delivery o
JOIN customers c ON o.customer_id = c.customer_id
LEFT JOIN order_revenue r ON o.order_id = r.order_id
LEFT JOIN latest_reviews rv ON o.order_id = rv.order_id AND rv.rn = 1;
"""

cursor.execute(master_sql)
conn.commit()
print("✓ Rebuilt master_orders with CTE")

# Step 3: Verify duplicates are gone
dups = pd.read_sql("""
    SELECT order_id, COUNT(*) AS n
    FROM master_orders
    GROUP BY order_id
    HAVING COUNT(*) > 1;
""", conn)
print(f"\nDuplicate orders after rebuild: {len(dups)}")

# Step 4: Verify total row count
total = pd.read_sql("SELECT COUNT(*) AS n FROM master_orders;", conn)
print(f"Total rows in master_orders: {total.iloc[0,0]}")

# Step 5: Spot-check the 5 originally-problematic orders
check = pd.read_sql("""
    SELECT order_id, COUNT(*) AS n
    FROM master_orders
    WHERE order_id IN (
        'df56136b8031ecd28e200bb18e6ddb2e',
        'c88b1d1b157a9999ce368f218a407141',
        '8e17072ec97ce29f0e1f111e598b0c85',
        '03c939fd7fd3b38f8485a0f95798f1f6',
        'ffaabba06c9d293a3c614e0515ddbabc'
    )
    GROUP BY order_id;
""", conn)
print(f"\nPreviously-problematic orders now appear:")
print(check)

✓ Dropped old master_orders
✓ Rebuilt master_orders with CTE

Duplicate orders after rebuild: 0
Total rows in master_orders: 96470

Previously-problematic orders now appear:
                           order_id  n
0  03c939fd7fd3b38f8485a0f95798f1f6  1
1  8e17072ec97ce29f0e1f111e598b0c85  1
2  c88b1d1b157a9999ce368f218a407141  1
3  df56136b8031ecd28e200bb18e6ddb2e  1
4  ffaabba06c9d293a3c614e0515ddbabc  1


In [21]:
# === Building customer_master from the cleaned master_orders ===

# Drop any existing version
cursor.execute("DROP TABLE IF EXISTS customer_master;")
conn.commit()

# Reference date for recency = max purchase date in dataset
max_date_query = pd.read_sql("SELECT MAX(order_purchase_timestamp) AS max_date FROM master_orders;", conn)
max_date = max_date_query.iloc[0, 0]
print(f"Reference date (max order date): {max_date}")

# Build the customer master table
customer_master_sql = f"""
CREATE TABLE customer_master AS
SELECT
    customer_unique_id,
    customer_state,

    -- FREQUENCY: total orders
    COUNT(DISTINCT order_id) AS total_orders,

    -- MONETARY: revenue
    ROUND(SUM(revenue), 2) AS total_revenue,
    ROUND(AVG(revenue), 2) AS avg_order_value,

    -- RECENCY: days since last purchase
    ROUND(julianday('{max_date}') - julianday(MAX(order_purchase_timestamp)), 0) AS recency_days,

    -- First and last order dates
    MIN(order_purchase_timestamp) AS first_order_date,
    MAX(order_purchase_timestamp) AS last_order_date,

    -- Customer lifespan in days
    ROUND(julianday(MAX(order_purchase_timestamp)) - julianday(MIN(order_purchase_timestamp)), 0) AS customer_lifespan_days,

    -- Delivery experience
    MAX(was_late) AS ever_had_late_delivery,
    SUM(was_late) AS num_late_deliveries,

    -- Average review score
    ROUND(AVG(review_score), 2) AS avg_review_score,

    -- Repeat customer flag
    CASE WHEN COUNT(DISTINCT order_id) > 1 THEN 1 ELSE 0 END AS is_repeat_customer

FROM master_orders
WHERE customer_unique_id IS NOT NULL
GROUP BY customer_unique_id, customer_state;
"""

cursor.execute(customer_master_sql)
conn.commit()
print("✓ Built customer_master")

# Verify
count = pd.read_sql("SELECT COUNT(*) AS n FROM customer_master;", conn)
print(f"Unique customers: {count.iloc[0,0]}")

# Distribution of orders per customer
orders_dist = pd.read_sql("""
SELECT total_orders, COUNT(*) AS num_customers,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM customer_master), 2) AS pct
FROM customer_master
GROUP BY total_orders
ORDER BY total_orders
LIMIT 10;
""", conn)
print("\nOrders per customer distribution:")
print(orders_dist)

Reference date (max order date): 2018-08-29 15:00:37
✓ Built customer_master
Unique customers: 93388

Orders per customer distribution:
   total_orders  num_customers    pct
0             1          90621  97.04
1             2           2542   2.72
2             3            179   0.19
3             4             27   0.03
4             5              9   0.01
5             6              5   0.01
6             7              3   0.00
7             9              1   0.00
8            15              1   0.00


In [23]:
# Overall summary
summary = pd.read_sql("""
SELECT
    COUNT(*) AS total_customers,
    SUM(is_repeat_customer) AS repeat_customers,
    ROUND(100.0 * SUM(is_repeat_customer) / COUNT(*), 2) AS pct_repeat,
    ROUND(AVG(total_revenue), 2) AS avg_clv,
    ROUND(AVG(recency_days), 0) AS avg_recency_days,
    ROUND(AVG(avg_review_score), 2) AS avg_review_score
FROM customer_master;
""", conn)
print("Customer-level summary:")
print(summary)

# Top states by customer count
states = pd.read_sql("""
SELECT customer_state,
       COUNT(*) AS customers,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM customer_master), 2) AS pct,
       ROUND(AVG(total_revenue), 2) AS avg_clv,
       ROUND(100.0 * SUM(is_repeat_customer) / COUNT(*), 2) AS pct_repeat
FROM customer_master
GROUP BY customer_state
ORDER BY customers DESC
LIMIT 10;
""", conn)
print("\nTop 10 states by customer count:")
print(states)

# Revenue distribution — are there outliers?
revenue_quartiles = pd.read_sql("""
SELECT
    MIN(total_revenue) AS min_rev,
    MAX(total_revenue) AS max_rev,
    ROUND(AVG(total_revenue), 2) AS mean_rev
FROM customer_master;
""", conn)
print("\nRevenue range:")
print(revenue_quartiles)

Customer-level summary:
   total_customers  repeat_customers  pct_repeat  avg_clv  avg_recency_days  \
0            93388              2767        2.96   141.56             238.0   

   avg_review_score  
0              4.15  

Top 10 states by customer count:
  customer_state  customers    pct  avg_clv  pct_repeat
0             SP      39149  41.92   129.42        3.10
1             RJ      11917  12.76   147.66        3.25
2             MG      11001  11.78   141.12        2.89
3             RS       5167   5.53   141.03        3.04
4             PR       4769   5.11   139.67        2.89
5             SC       3449   3.69   147.00        2.61
6             BA       3158   3.38   156.30        2.79
7             DF       2019   2.16   146.85        2.87
8             ES       1928   2.06   139.34        2.90
9             GO       1895   2.03   149.25        3.06

Revenue range:
   min_rev  max_rev  mean_rev
0     0.85  13440.0    141.56
